In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import os
import missingno as msno

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,VotingClassifier,GradientBoostingClassifier,BaggingClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('aug_train.csv')

In [4]:
df.shape

(19158, 14)

In [12]:
df.dtypes.value_counts()

object     10
int64       2
float64     2
Name: count, dtype: int64

In [3]:
df.head()

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [7]:
df.tail()

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
19153,7386,city_173,0.878,Male,No relevent experience,no_enrollment,Graduate,Humanities,14,NaN,NaN,1,42,1.0
19154,31398,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,14,NaN,NaN,4,52,1.0
19155,24576,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,50-99,Pvt Ltd,4,44,0.0
19156,5756,city_65,0.802,Male,Has relevent experience,no_enrollment,High School,NaN,<1,500-999,Pvt Ltd,2,97,0.0
19157,23834,city_67,0.855,NaN,No relevent experience,no_enrollment,Primary School,NaN,2,NaN,NaN,1,127,0.0


In [9]:
df.sample(3)

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
14348,22444,city_27,0.848,Male,Has relevent experience,no_enrollment,Graduate,STEM,12,<10,Funded Startup,>4,4,0.0
8372,5315,city_21,0.624,Male,No relevent experience,Full time course,Graduate,STEM,1,NaN,Pvt Ltd,never,94,0.0
15413,26624,city_16,0.910,Male,Has relevent experience,no_enrollment,Graduate,STEM,16,10/49,Pvt Ltd,3,202,0.0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   enrollee_id             19158 non-null  int64  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevent_experience     19158 non-null  object 
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int64  
 13  target                  19158 non-null  float64
dtypes: float64(2), int64(2), object(10)
me

In [20]:
df.isna().sum().sort_values(ascending=False)/len(df)*100

company_type              32.049274
company_size              30.994885
gender                    23.530640
major_discipline          14.683161
education_level            2.401086
last_new_job               2.207955
enrolled_university        2.014824
experience                 0.339284
enrollee_id                0.000000
city                       0.000000
relevent_experience        0.000000
city_development_index     0.000000
training_hours             0.000000
target                     0.000000
dtype: float64

In [ ]:
df.describe(include='all') #<-- NEW THING 

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
count,19158.000000,19158,19158.000000,14650,19158,18772,18698,16345,19093,13220,13018,18735,19158.000000,19158.000000
unique,NaN,123,NaN,3,2,3,5,6,22,8,6,6,NaN,NaN
top,NaN,city_103,NaN,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,50-99,Pvt Ltd,1,NaN,NaN
freq,NaN,4355,NaN,13221,13792,13817,11598,14492,3286,3083,9817,8040,NaN,NaN
mean,16875.358179,NaN,0.828848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.366896,0.249348
std,9616.292592,NaN,0.123362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.058462,0.432647
min,1.000000,NaN,0.448000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000
25%,8554.250000,NaN,0.740000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.000000,0.000000
50%,16982.500000,NaN,0.903000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.000000,0.000000
75%,25169.750000,NaN,0.920000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.000000,0.000000


In [24]:
# Too Early to call skew corr and kurt ; must be called later 

In [ ]:
# groupby Intution 
''' 
Example 1: Useless Column
pythondf.groupby('gender')['churn'].mean()

# Output:
# Male:   0.27
# Female: 0.28
# Difference: 0.01 (1%)

# Overall churn rate: 0.27
What this tells you:

Whether customer is Male or Female, ~27% churn
Gender provides NO INFORMATION about churn
Model will learn: "gender doesn't matter"

If you encode and include it:
python# After encoding: gender_Male (0 or 1)
# Model learns: coefficient ≈ 0.01 (nearly zero)
# You wasted computation encoding a useless feature

Example 2: Powerful Column
pythondf.groupby('contract_type')['churn'].mean()

# Output:
# Month-to-month: 0.85  (85% churn!)
# One year:       0.15  (15% churn)
# Two year:       0.05  (5% churn)
# Difference: 0.80 (80%!)

# Overall churn rate: 0.35
What this tells you:

Contract type STRONGLY predicts churn
Month-to-month customers churn 17x more than two-year!
This MUST be encoded and included
'''

In [48]:
grp_exp_target = df.groupby('relevent_experience')['target']

In [54]:
for x in df.select_dtypes('object').drop(columns=['city','experience']).columns:
    grouping = df.groupby(x)['target']
    print(f'The mean of the group : {grouping.mean()}\n')
    print(f'The std of the group : {grouping.std()}\n')


The mean of the group : gender
Female    0.263328
Male      0.227819
Other     0.261780
Name: target, dtype: float64

The std of the group : gender
Female    0.440617
Male      0.419441
Other     0.440759
Name: target, dtype: float64

The mean of the group : relevent_experience
Has relevent experience    0.214690
No relevent experience     0.338427
Name: target, dtype: float64

The std of the group : relevent_experience
Has relevent experience    0.410622
No relevent experience     0.473219
Name: target, dtype: float64

The mean of the group : enrolled_university
Full time course    0.380889
Part time course    0.252087
no_enrollment       0.211406
Name: target, dtype: float64

The std of the group : enrolled_university
Full time course    0.485670
Part time course    0.434392
no_enrollment       0.408321
Name: target, dtype: float64

The mean of the group : education_level
Graduate          0.279790
High School       0.195340
Masters           0.214400
Phd               0.140097
Prima

In [ ]:
'''
I think the usefull columns are : 

relevent_experience , enrolled_university , education_level ,
 major_discipline , company_size , company_type ,  last_new_job
( everything expect gender ) 

'''

# This means gender doesnt contributes that much to our model

In [ ]:
# IMPROVEMENT
'''
def analyze_categorical_columns(df, target_col='target', exclude_cols=None):
    """
    Pre-analysis: Check which categorical columns are worth keeping
    """
    if exclude_cols is None:
        exclude_cols = []
    
    categorical_cols = df.select_dtypes('object').columns
    categorical_cols = [col for col in categorical_cols if col not in exclude_cols]
    
    results = []
    
    print("="*70)
    print("CATEGORICAL COLUMN ANALYSIS")
    print("="*70)
    
    for col in categorical_cols:
        print(f"\n📊 Column: {col}")
        print("-" * 70)
        
        # Group by category
        grouped = df.groupby(col)[target_col].agg(['count', 'mean', 'std'])
        print(grouped.sort_values('mean', ascending=False))
        
        # Calculate useful metrics
        overall_mean = df[target_col].mean()
        max_diff = grouped['mean'].max() - grouped['mean'].min()
        mean_std = grouped['mean'].std()
        
        # Decision
        is_useful = max_diff > 0.05  # Categories differ by >5%
        
        print(f"\n📈 Overall target mean: {overall_mean:.3f}")
        print(f"📈 Max difference between categories: {max_diff:.3f}")
        print(f"📈 Variance in category means: {mean_std:.3f}")
        
        if is_useful:
            print(f"✅ VERDICT: USEFUL - Keep this column")
        else:
            print(f"❌ VERDICT: LOW SIGNAL - Consider dropping")
        
        results.append({
            'column': col,
            'num_categories': df[col].nunique(),
            'max_diff': max_diff,
            'is_useful': is_useful
        })
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    summary_df = pd.DataFrame(results).sort_values('max_diff', ascending=False)
    print(summary_df)
    
    print(f"\n✅ Useful columns: {summary_df[summary_df['is_useful']]['column'].tolist()}")
    print(f"❌ Consider dropping: {summary_df[~summary_df['is_useful']]['column'].tolist()}")
    
    return summary_df

# Usage:
results = analyze_categorical_columns(df, target_col='churn', exclude_cols=['city', 'experience'])
```

---

## **What This Improved Version Does:**

1. **Shows WHICH column** you're analyzing
2. **Shows the actual categories** and their counts
3. **Gives a verdict** - keep or drop?
4. **Returns a summary** you can use for decisions
5. **No hardcoded drops** - flexible for any dataset
6. **Sorted by importance** - see best features first

---

## **Example Output:**
```
======================================================================
CATEGORICAL COLUMN ANALYSIS
======================================================================

📊 Column: contract_type
----------------------------------------------------------------------
                    count      mean       std
contract_type                               
Month-to-month        100  0.900000  0.302765
One year               50  0.100000  0.303046
Two year               20  0.050000  0.223607

📈 Overall target mean: 0.350
📈 Max difference between categories: 0.850
📈 Variance in category means: 0.457
✅ VERDICT: USEFUL - Keep this column

📊 Column: gender
----------------------------------------------------------------------
           count      mean       std
gender                              
Male        85   0.352941  0.480384
Female      85   0.347059  0.478746

📈 Overall target mean: 0.350
📈 Max difference between categories: 0.006
📈 Variance in category means: 0.004
❌ VERDICT: LOW SIGNAL - Consider dropping

======================================================================
SUMMARY
======================================================================
          column  num_categories  max_diff  is_useful
0  contract_type               3     0.850       True
1         gender               2     0.006      False

✅ Useful columns: ['contract_type']
❌ Consider dropping: ['gender']
'''

In [55]:
# Numeric Value Analysis 

In [65]:
numeric_columns = df.select_dtypes(include = ['int','float']).drop(columns=['enrollee_id']).columns

In [70]:
df[numeric_columns].corr(numeric_only=True)['target']

city_development_index   -0.341665
training_hours           -0.021577
target                    1.000000
Name: target, dtype: float64

In [68]:
df[numeric_columns].skew(numeric_only=True)

city_development_index   -0.995428
training_hours            1.819237
target                    1.158815
dtype: float64

In [69]:
df[numeric_columns].kurt(numeric_only=True)

city_development_index   -0.538532
training_hours            3.840539
target                   -0.657217
dtype: float64